In [1]:
!pip install anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 10.0 MB/s eta 0:00:00


In [ ]:
import anthropic
import pandas as pd
import time
from tqdm import tqdm

# 1. Inisialisasi client
client = anthropic.Anthropic(api_key="ISI API KEY
# 2. Fungsi anotasi batch
def annotate_batch(texts):
    joined_texts = "\n".join([f"{i+1}. {t}" for i, t in enumerate(texts)])
    prompt = f"""
    Berikut adalah beberapa teks ulasan pinjaman online.
    Untuk setiap teks, klasifikasikan emosi utama ke dalam salah satu kategori:
    [Marah, Senang, Kecewa, Takut].

    Format jawaban:
    1. Marah
    2. Senang
    3. Kecewa
    4. Takut
    ...

    Teks:
    {joined_texts}
    """

    message = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text.strip().splitlines()

# 3. Load dataset
df = pd.read_csv("/content/14k_realll_zahraa.csv")

# 4. Proses batch dengan tqdm
batch_size = 20
emosi = []

for i in tqdm(range(0, len(df), batch_size), desc="Annotating"):
    batch = df["clean_content"].iloc[i:i+batch_size].tolist()
    results = annotate_batch(batch)
    for r in results:
        if "." in r:
            emosi.append(r.split(".", 1)[1].strip())
        else:
            emosi.append(r.strip())
    time.sleep(1)

# 5. Tambahkan ke dataframe
df["emosi"] = emosi

# 6. Simpan hasil
df.to_csv("data_with_emosi.csv", index=False)
print("✅")


Annotating:   2%|▏         | 16/746 [01:23<1:01:09,  5.03s/it]